# Phase 3: SHAP値を用いたクラスタリング分析

**目的**: Phase 2で得たSHAP値を使って楽曲をグループ化し、「なぜ人気か/不人気か」のパターンで楽曲を分類・深掘りする  
**データ**: Kaggle Spotify 1 Million Tracks（約116万曲、2000〜2023年）  
**設計**: クラスタリングはSHAP値で実施（音楽的類似性ではなく人気度への影響パターンでグループ化）

## Step 0: セットアップ・データ読み込み・モデル学習

In [ ]:
# shap はColab環境に標準でインストール済み
%pip install japanize-matplotlib --quiet

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile
import os

boxurl = "https://tus.box.com/shared/static/491ie3a5kdgg7hfajivmwlou6zq2fnzj.zip"
filename = "Kaggle_Spotify_1Million_Tracks.zip"

if not os.path.exists("/content/spotify_data.csv"):
    print("データをダウンロード中...")
    urlretrieve(boxurl, filename)
    ZipFile(filename).extractall()
    print("完了")
else:
    print("データは既に存在します")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import japanize_matplotlib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams['axes.unicode_minus'] = False

print(f"shap バージョン: {shap.__version__}")

In [ ]:
# データ読み込み・前処理
df = pd.read_csv("/content/spotify_data.csv", index_col=0)
print(f"データ形状: {df.shape}")

df_clean = df.dropna(subset=['popularity']).copy()

# アーティスト出現頻度（アーティスト人気の代理変数）
artist_freq = df_clean['artist_name'].value_counts()
df_clean['artist_freq'] = df_clean['artist_name'].map(artist_freq)

# ジャンルをラベルエンコーディング
le = LabelEncoder()
df_clean['genre_enc'] = le.fit_transform(df_clean['genre'].astype(str))

FEATURE_COLS = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence',
    'tempo', 'duration_ms', 'key', 'mode', 'time_signature',
    'year', 'artist_freq', 'genre_enc'
]

X = df_clean[FEATURE_COLS]
y = df_clean['popularity']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"訓練: {len(X_train):,} 件 / 検証: {len(X_val):,} 件 / テスト: {len(X_test):,} 件")

In [ ]:
# GradientBoosting モデル学習（Phase 2と同一パラメータ）
print("HistGradientBoosting 学習中...")
gb = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_depth=8,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42
)
t0 = time.time()
gb.fit(X_train, y_train)
gb_time = time.time() - t0

gb_pred = gb.predict(X_test)
print(f"完了（{gb_time:.1f}秒）")
print(f"  MAE: {mean_absolute_error(y_test, gb_pred):.4f}")
print(f"  R²:  {r2_score(y_test, gb_pred):.4f}")

In [ ]:
# SHAP Explainer 構築（Phase 2と同一）
print("SHAP Explainer を構築中...")
rng = np.random.default_rng(42)
bg_idx = rng.choice(len(X_train), size=200, replace=False)
X_bg = X_train.iloc[bg_idx]

explainer_gb = shap.Explainer(gb, X_bg)
print("完了")

## Step 1: データ絞り込み（4条件の比較）

In [ ]:
print("=" * 60)
print("Step 1: データ絞り込み条件の比較")
print("=" * 60)

all_std = df_clean['popularity'].std()
print(f"全データ: {len(df_clean):,} 件 / popularity std: {all_std:.2f}\n")

conditions = {
    'cond_A': df_clean[df_clean['popularity'] >= 40],
    'cond_B': df_clean[df_clean['year'] >= 2015],
    'cond_C': df_clean[(df_clean['popularity'] >= 40) & (df_clean['year'] >= 2015)],
    'cond_D': df_clean[df_clean['genre'].str.contains('pop', case=False, na=False)],
}

cond_labels = {
    'cond_A': 'popularity >= 40',
    'cond_B': 'year >= 2015',
    'cond_C': 'popularity >= 40 かつ year >= 2015',
    'cond_D': "genre に 'pop' を含む",
}

stats = {}
for cond_id, df_cond in conditions.items():
    n = len(df_cond)
    pop_mean   = df_cond['popularity'].mean()
    pop_median = df_cond['popularity'].median()
    pop_std    = df_cond['popularity'].std()
    top5_genre = df_cond['genre'].value_counts().head(5)
    stats[cond_id] = {'n': n, 'mean': pop_mean, 'median': pop_median, 'std': pop_std}

    print(f"【{cond_id}】{cond_labels[cond_id]}")
    print(f"  件数: {n:,}")
    print(f"  popularity: 平均={pop_mean:.1f}, 中央値={pop_median:.1f}, std={pop_std:.2f}")
    print(f"  上位5ジャンル:")
    for genre, cnt in top5_genre.items():
        print(f"    {genre}: {cnt:,}件 ({cnt/n*100:.1f}%)")
    print()

In [ ]:
# 条件選択: 5,000〜50,000件 かつ std < 全データstd を優先
valid = [(cid, s['n'], s['std'])
         for cid, s in stats.items()
         if 5000 <= s['n'] <= 50000 and s['std'] < all_std]

if valid:
    selected_cond = sorted(valid, key=lambda x: x[2])[0][0]  # stdが最小のものを選択
    reason = f"件数が{stats[selected_cond]['n']:,}件（5,000〜50,000件の範囲内）かつstd={stats[selected_cond]['std']:.2f}が全データより小さい"
else:
    # 範囲外の場合: 件数が50,000件に最も近いものを選択
    selected_cond = min(stats.keys(), key=lambda c: abs(stats[c]['n'] - 50000))
    reason = "件数が選択基準範囲外のため、50,000件に最も近い条件を選択"

df_target = conditions[selected_cond].copy()

print(f"▶ 選択した条件: {selected_cond}（{cond_labels[selected_cond]}）")
print(f"  件数: {len(df_target):,}")
print(f"  理由: {reason}")

## Step 2: SHAP値の計算（絞り込み後データに対して）

In [ ]:
print("Step 2: SHAP値の計算")

N_MAX = 10000
rng2 = np.random.default_rng(42)

if len(df_target) > N_MAX:
    idx_sample = rng2.choice(len(df_target), size=N_MAX, replace=False)
    df_target_sample = df_target.iloc[idx_sample].reset_index(drop=True)
    print(f"サンプリング: {len(df_target):,} → {N_MAX:,} 件")
else:
    df_target_sample = df_target.reset_index(drop=True)
    print(f"全件使用: {len(df_target_sample):,} 件")

X_target_sample = df_target_sample[FEATURE_COLS]

print("SHAP値を計算中（数分かかる場合があります）...")
shap_values_target = explainer_gb(X_target_sample, check_additivity=False)
shap_matrix = shap_values_target.values  # shape: (n, 16)
print(f"完了 / shap_matrix shape: {shap_matrix.shape}")

## Step 3: クラスタリング（KMeans + 最適クラスタ数の探索）

In [ ]:
# Step 3-1: StandardScaler でSHAP値を標準化
print("Step 3-1: SHAP値の標準化")
scaler = StandardScaler()
shap_scaled = scaler.fit_transform(shap_matrix)
print(f"標準化完了 / shape: {shap_scaled.shape}")

In [ ]:
# Step 3-2: 最適クラスタ数の探索（エルボー法 + シルエットスコア）
print("Step 3-2: 最適クラスタ数の探索（k=2〜10）")

inertias    = []
silhouettes = []
K_range     = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(shap_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(
        shap_scaled, labels_k,
        sample_size=min(2000, len(shap_scaled)),
        random_state=42
    )
    silhouettes.append(sil)
    print(f"  k={k}: 慣性={km.inertia_:.1f}, シルエット={sil:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('クラスタ数 k')
axes[0].set_ylabel('慣性（Inertia）')
axes[0].set_title('エルボー法')
axes[0].grid(alpha=0.3)

best_k = list(K_range)[int(np.argmax(silhouettes))]

axes[1].plot(list(K_range), silhouettes, 'o-', color='darkorange', linewidth=2)
axes[1].axvline(x=best_k, color='red', linestyle='--', alpha=0.7, label=f'best k={best_k}')
axes[1].set_xlabel('クラスタ数 k')
axes[1].set_ylabel('シルエットスコア')
axes[1].set_title('シルエットスコア')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('最適クラスタ数の探索', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('phase3_optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n最適クラスタ数: k = {best_k}（シルエットスコア: {max(silhouettes):.3f}）")

In [ ]:
# Step 3-3: best_k で最終クラスタリング
print(f"Step 3-3: k={best_k} でクラスタリング実行")

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(shap_scaled)
df_target_sample['cluster'] = cluster_labels

print("クラスタリング完了 / クラスタ別件数:")
print(df_target_sample['cluster'].value_counts().sort_index().to_string())

## Step 4: 各クラスタの特徴量分析

In [ ]:
# Step 4-1: クラスタ概要表
print("Step 4-1: クラスタ概要")
print("=" * 60)

cluster_summary = []
for k in range(best_k):
    mask = df_target_sample['cluster'] == k
    df_k = df_target_sample[mask]
    top3_genre = df_k['genre'].value_counts().head(3).index.tolist()

    cluster_summary.append({
        'クラスタ':         k,
        '件数':             len(df_k),
        'popularity_mean':  round(df_k['popularity'].mean(), 2),
        'popularity_std':   round(df_k['popularity'].std(),  2),
        'year_median':      int(df_k['year'].median()),
        'top3_genre':       ', '.join(top3_genre),
    })

    print(f"クラスタ {k}:")
    print(f"  件数: {len(df_k):,}")
    print(f"  popularity: 平均={df_k['popularity'].mean():.1f}, std={df_k['popularity'].std():.2f}")
    print(f"  リリース年 中央値: {int(df_k['year'].median())}")
    print(f"  上位ジャンル: {', '.join(top3_genre)}")
    print()

cluster_summary_df = pd.DataFrame(cluster_summary)
print(cluster_summary_df.to_string(index=False))

In [ ]:
# Step 4-2: レーダーチャート
N_feat = len(FEATURE_COLS)
angles = np.linspace(0, 2 * np.pi, N_feat, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))
colors = plt.cm.tab10(np.linspace(0, 1, best_k))

for k in range(best_k):
    mask = df_target_sample['cluster'] == k
    mean_shap = shap_matrix[mask].mean(axis=0)
    values = mean_shap.tolist() + [mean_shap[0]]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'クラスタ {k}', color=colors[k])
    ax.fill(angles, values, alpha=0.08, color=colors[k])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(FEATURE_COLS, fontsize=9)
ax.set_title('クラスタ別 平均SHAP値 レーダーチャート', fontsize=13, fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('phase3_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 4-3: ヒートマップ
heatmap_data = np.zeros((best_k, len(FEATURE_COLS)))
for k in range(best_k):
    mask = df_target_sample['cluster'] == k
    heatmap_data[k] = shap_matrix[mask].mean(axis=0)

heatmap_df = pd.DataFrame(
    heatmap_data,
    index=[f'クラスタ {k}' for k in range(best_k)],
    columns=FEATURE_COLS
)

fig, ax = plt.subplots(figsize=(16, max(4, best_k * 1.5)))
sns.heatmap(
    heatmap_df,
    annot=True, fmt='.3f',
    cmap='RdBu_r', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('クラスタ別 平均SHAP値 ヒートマップ（赤=正、青=負）', fontsize=13, fontweight='bold')
ax.set_xlabel('特徴量')
ax.set_ylabel('クラスタ')

plt.tight_layout()
plt.savefig('phase3_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5: 知っている曲のクラスタを特定・深掘り

In [ ]:
# Step 5-1: 著名アーティストの楽曲検索
print("Step 5-1: 著名アーティストの楽曲検索")
print("=" * 60)

famous_artists = [
    'Taylor Swift', 'Ed Sheeran', 'Billie Eilish', 'Ariana Grande',
    'Drake', 'The Weeknd', 'Post Malone', 'Justin Bieber',
    'Coldplay', 'Bruno Mars'
]

artist_results = {}
for artist in famous_artists:
    mask = df_target_sample['artist_name'].str.contains(artist, case=False, na=False)
    df_artist = df_target_sample[mask].sort_values('popularity', ascending=False)
    top3 = [(row['track_name'], int(row['popularity']), int(row['cluster']))
            for _, row in df_artist.head(3).iterrows()]
    artist_results[artist] = {'count': len(df_artist), 'top3': top3}

    print(f"アーティスト名: {artist}")
    print(f"  → 該当件数: {len(df_artist)}曲")
    if top3:
        print(f"  → 代表曲（popularity上位3件）:")
        for track, pop, cl in top3:
            print(f"     [{str(track)[:40]}, popularity={pop}, クラスタ={cl}]")
    else:
        print(f"  → 該当なし（サンプル内に含まれていない可能性あり）")
    print()

In [ ]:
# Step 5-2: 注目クラスタの選定
print("Step 5-2: 注目クラスタの選定")

cluster_famous_count = {k: 0 for k in range(best_k)}
for artist, result in artist_results.items():
    for track, pop, cl in result['top3']:
        cluster_famous_count[cl] += 1

cluster_pop_mean = {
    k: df_target_sample[df_target_sample['cluster'] == k]['popularity'].mean()
    for k in range(best_k)
}

print("クラスタ別 著名楽曲数 / popularity平均:")
for k in range(best_k):
    print(f"  クラスタ {k}: 著名楽曲={cluster_famous_count[k]}曲 / popularity平均={cluster_pop_mean[k]:.1f}")

focus_cluster = max(range(best_k),
                    key=lambda k: (cluster_famous_count[k], cluster_pop_mean[k]))

print(f"\n▶ 注目クラスタ: クラスタ {focus_cluster}")
print(f"  著名楽曲数: {cluster_famous_count[focus_cluster]}曲")
print(f"  popularity平均: {cluster_pop_mean[focus_cluster]:.1f}")

In [ ]:
# Step 5-3a: 注目クラスタ内の著名楽曲一覧（popularity降順 上位10曲）
print(f"Step 5-3: 注目クラスタ（クラスタ {focus_cluster}）の詳細分析")
print("=" * 60)

df_focus = df_target_sample[df_target_sample['cluster'] == focus_cluster].copy()

famous_pattern = '|'.join(famous_artists)
famous_mask = df_focus['artist_name'].str.contains(famous_pattern, case=False, na=False)
df_famous_in_focus = df_focus[famous_mask].sort_values('popularity', ascending=False)

print(f"■ クラスタ {focus_cluster} 内の著名楽曲 TOP10（popularity降順）")
if len(df_famous_in_focus) > 0:
    display_df = df_famous_in_focus.head(10)[['track_name', 'artist_name', 'popularity', 'year', 'genre']]
else:
    print("  著名アーティストの楽曲がサンプル内に見つからないため、popularity上位10曲を表示")
    display_df = df_focus.sort_values('popularity', ascending=False).head(10)[['track_name', 'artist_name', 'popularity', 'year', 'genre']]

print(display_df.to_string(index=False))

In [ ]:
# Step 5-3b: Waterfall Plot（最高・最低 popularity 楽曲）
print(f"\n■ Waterfall Plot（クラスタ {focus_cluster} の高・低人気楽曲）")

focus_indices = df_target_sample[df_target_sample['cluster'] == focus_cluster].index.tolist()
focus_pop     = df_target_sample.loc[focus_indices, 'popularity']
idx_high_pop  = focus_pop.idxmax()
idx_low_pop   = focus_pop.idxmin()

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for ax, idx, label in [
    (axes[0], idx_high_pop, '最高人気楽曲'),
    (axes[1], idx_low_pop,  '最低人気楽曲（例外候補）'),
]:
    row   = df_target_sample.loc[idx]
    title = (f'{label}\n'
             f'{str(row["track_name"])[:35]} / {str(row["artist_name"])[:20]}\n'
             f'popularity={int(row["popularity"])}')
    plt.sca(ax)
    shap.plots.waterfall(shap_values_target[idx], show=False)
    ax.set_title(title, fontsize=10)

plt.suptitle(f'クラスタ {focus_cluster} - Waterfall Plot 比較', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'phase3_waterfall_cluster{focus_cluster}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 5-3c: 例外楽曲の発見（重心からの距離）
print(f"\n■ 例外楽曲の発見（クラスタ {focus_cluster} 内）")

centroid  = shap_matrix[focus_indices].mean(axis=0)
distances = np.linalg.norm(shap_matrix[focus_indices] - centroid, axis=1)

df_focus_dist              = df_target_sample.loc[focus_indices].copy()
df_focus_dist['dist']      = distances
df_exceptions              = df_focus_dist.sort_values('dist', ascending=False).head(5)

print("重心からの距離が大きい楽曲（例外楽曲）TOP5:")
for _, row in df_exceptions.iterrows():
    idx      = row.name
    diff     = shap_matrix[idx] - centroid
    top_i    = int(np.argmax(np.abs(diff)))
    top_feat = FEATURE_COLS[top_i]
    diff_val = diff[top_i]
    direction = "クラスタ平均より高い（人気を押し上げ）" if diff_val > 0 else "クラスタ平均より低い（人気を押し下げ）"

    print(f"\n  曲名: {str(row['track_name'])[:40]}")
    print(f"  アーティスト: {row['artist_name']} / popularity: {int(row['popularity'])}")
    print(f"  重心からの距離: {row['dist']:.3f}")
    print(f"  最も異質な特徴量: {top_feat}（{direction}）")

In [ ]:
# Step 5-3d: popularity 分布（注目クラスタ vs その他）
print(f"\n■ popularity 分布: 注目クラスタ {focus_cluster} vs その他")

fig, ax = plt.subplots(figsize=(12, 6))
colors_dist = plt.cm.tab10(np.linspace(0, 1, best_k))

for k in range(best_k):
    mask   = df_target_sample['cluster'] == k
    label  = f'クラスタ {k}（注目）' if k == focus_cluster else f'クラスタ {k}'
    alpha  = 0.85 if k == focus_cluster else 0.4
    zorder = 3 if k == focus_cluster else 2
    ax.hist(
        df_target_sample[mask]['popularity'],
        bins=30, alpha=alpha, label=label,
        color=colors_dist[k], edgecolor='white',
        density=True, zorder=zorder
    )

ax.set_xlabel('Popularity スコア')
ax.set_ylabel('密度')
ax.set_title(f'クラスタ別 popularity 分布（注目: クラスタ {focus_cluster}）',
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('phase3_popularity_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6: まとめ・考察出力

In [ ]:
print("=" * 54)
print("         Phase 3 分析まとめ")
print("=" * 54)

print(f"\n■ データ絞り込み条件: {selected_cond}（{cond_labels[selected_cond]}）")
print(f"  理由: {reason}")
print(f"■ 分析対象件数: {len(df_target_sample):,} 件")

print(f"\n■ クラスタリング結果:")
print(f"  最適クラスタ数: best_k = {best_k}")
print(f"  （シルエットスコア: {max(silhouettes):.3f}）")

print(f"\n■ 各クラスタ概要:")
for row in cluster_summary:
    k = row['クラスタ']
    print(f"  クラスタ{k}: {row['件数']:,}件 / popularity平均{row['popularity_mean']:.1f} / 主要ジャンル: {row['top3_genre']}")

focus_shap_mean = shap_matrix[df_target_sample['cluster'] == focus_cluster].mean(axis=0)
top_idx         = np.argsort(focus_shap_mean)[::-1]

print(f"\n■ 注目クラスタ（クラスタ {focus_cluster}）の特徴:")
print(f"  - SHAP値が特に高い特徴量: {FEATURE_COLS[top_idx[0]]}（平均SHAP値: {focus_shap_mean[top_idx[0]]:.3f}）")
print(f"  - SHAP値が特に低い特徴量: {FEATURE_COLS[top_idx[-1]]}（平均SHAP値: {focus_shap_mean[top_idx[-1]]:.3f}）")

rep_tracks = [(t, a) for art, res in artist_results.items()
              for t, p, c in res['top3'] if c == focus_cluster
              for a in [art]]
if rep_tracks:
    print(f"  - 代表的な著名楽曲: {rep_tracks[0][0]} / {rep_tracks[0][1]}")

print(f"\n■ 発見された例外楽曲:")
for _, row in df_exceptions.head(3).iterrows():
    idx      = row.name
    diff     = shap_matrix[idx] - centroid
    top_feat = FEATURE_COLS[int(np.argmax(np.abs(diff)))]
    print(f"  - {str(row['track_name'])[:35]}: {top_feat} のSHAP値がクラスタ平均から大きく乖離")

print("\n" + "=" * 54)